# Gate 0 per drive: map-match a new drive and count its clean segments

**Run once per new drive:** set `SESSION` in Cell 1 to the footage folder name, then **Restart & Run All**.
The pilot analysis (stale-label discovery, city-wide routing) stays in `gate0_map_match.ipynb`.

- **Cells 1-4:** load the street segments and the drive's GPS track, map them, snap each GPS fix to a street, and score the drive.
- **Cell 5:** keep only *clean* segments (have a 2023 PCI and no repair since Feb 2023), flag alleys and the
  **primary** analysis set (clean, residential, non-alley), then total them across every drive so far
  (Viability check V5, plus the primary count that V6/V7 require).

**Reads:** `data/raw/2026-10-24sd_paving_segs_datasd.geojson`, `data/raw/completed_paving_datasd.csv`, `data/processed/segment_cpname.csv`, `data/processed/segments_clean.csv`, `data/gps_logs/<SESSION>/_track.csv`
**Writes:** `reports/figures/drive_<SESSION>.html`, `data/processed/gate0_segments_<SESSION>.csv`, `gate0_fixes_<SESSION>.csv`, `clean_driven_segments.csv`

In [1]:
# Gate 0 — Cell 1: load the street segments (with recovered cpname) and today's GPS track
from pathlib import Path
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd()
if ROOT.name == "02_data_cleaning":
    ROOT = ROOT.parent
SESSION = "11-11-2026-Pacific-Beach"      # <- change per drive

seg = gpd.read_file(ROOT / "data/raw/2026-10-24sd_paving_segs_datasd.geojson")
seg["pci23"]   = pd.to_numeric(seg["pci23"], errors="coerce")
seg["seg_key"] = seg["iamfloc"].astype("string").str.strip().str.upper()

cp = pd.read_csv(ROOT / "data/processed/segment_cpname.csv").dropna(subset=["seg_key"])
cp["seg_key"] = cp["seg_key"].astype("string")
n0  = len(seg)
seg = seg.drop(columns="cpname").merge(cp, on="seg_key", how="left")
assert len(seg) == n0, "merge duplicated rows — stop and check"

trk = pd.read_csv(ROOT / f"data/gps_logs/{SESSION}/_track.csv")
trk = gpd.GeoDataFrame(trk, geometry=gpd.points_from_xy(trk["longitude"], trk["latitude"]),
                       crs="EPSG:4326")
print(f"{len(seg):,} segments ({seg['cpname'].notna().sum():,} with cpname) | "
      f"{len(trk):,} GPS fixes, {trk['local'].iloc[0]} -> {trk['local'].iloc[-1]}")

32,745 segments (32,710 with cpname) | 2,291 GPS fixes, 2026-11-11 12:40:09 -> 2026-11-11 13:19:16


In [2]:
# Gate 0 — Cell 2: where did I drive? GPS fixes over the 2023 PCI map
minx, miny, maxx, maxy = trk.to_crs(32611).total_bounds          # track extent, in meters
near = seg.to_crs(32611).cx[minx - 200 : maxx + 200, miny - 200 : maxy + 200]

m = near.explore(column="pci23", cmap="RdYlGn", vmin=0, vmax=100, tiles="OpenStreetMap",
                 tooltip=["rd20full", "pci23", "pci23_d", "cpname"],
                 style_kwds={"weight": 4}, legend=True)
trk.explore(m=m, color="blue", marker_kwds={"radius": 3},
            tooltip=["local", "speed_mph", "track_deg"])
out = ROOT / f"reports/figures/drive_{SESSION}.html"
m.save(str(out))
print(f"{len(near):,} segments in the drive area | saved: {out}")
m

In [3]:
# Gate 0 — Cell 3: snap each GPS fix to a street segment (distance + axial heading), label why fixes drop
import numpy as np
SNAP_M, HEAD_MAX, END_BUF_M = 20, 30, 15     # tunable: max snap distance (m), max heading gap (deg), end trim (m)

seg_m = seg[seg["seg_key"].notna()].to_crs(32611)     # meters (UTM 11N)
trk_m = trk.to_crs(32611)

# 1) candidates: every segment within SNAP_M of each fix
buf   = gpd.GeoDataFrame(geometry=trk_m.buffer(SNAP_M), crs=trk_m.crs)
pairs = gpd.sjoin(buf, seg_m[["geometry"]], predicate="intersects").reset_index(names="fix")

# 2) distance, position along the segment, and the segment's direction at that spot
P = trk_m.geometry.loc[pairs["fix"]].reset_index(drop=True)
L = seg_m.geometry.loc[pairs["index_right"]].reset_index(drop=True)
pairs["dist_m"]    = P.distance(L).to_numpy()
s                  = L.project(P).to_numpy()            # meters along the segment
pairs["along_m"]   = s
pairs["seg_len_m"] = L.length.to_numpy()
a, b    = L.interpolate(np.clip(s - 3, 0, None)), L.interpolate(s + 3)   # 3 m either side
seg_brg = np.degrees(np.arctan2(b.x.to_numpy() - a.x.to_numpy(), b.y.to_numpy() - a.y.to_numpy())) % 360
d = np.abs((trk_m["track_deg"].loc[pairs["fix"]].to_numpy() - seg_brg + 180) % 360 - 180)  # 0..180
pairs["head_diff"] = np.minimum(d, 180 - d)             # axial: either driving direction counts

# 3) best match per fix = closest candidate whose direction agrees
best = (pairs[pairs["head_diff"] <= HEAD_MAX].sort_values("dist_m")
        .drop_duplicates("fix").set_index("fix"))
fx = trk_m[["clip", "local", "speed_mph", "track_deg"]].join(
        best[["index_right", "dist_m", "head_diff", "along_m", "seg_len_m"]])
fx["seg_key"] = fx["index_right"].map(seg_m["seg_key"])

# 4) one reason per fix (first rule that applies wins)
fx["reason"] = np.select(
    [~fx["speed_mph"].between(3, 60),
     ~fx.index.isin(pairs["fix"]),
     fx["index_right"].isna(),
     (fx["along_m"] < END_BUF_M) | (fx["along_m"] > fx["seg_len_m"] - END_BUF_M)],
    ["speed_gate", "no_segment_within_20m", "heading_mismatch", "near_intersection"],
    default="ok")

print(fx["reason"].value_counts().to_string(), "\n")
ok = fx["reason"] == "ok"
print(f"usable fixes: {ok.sum():,} of {len(fx):,} ({ok.mean():.0%})")
print(f"snap distance on usable fixes: median {fx.loc[ok, 'dist_m'].median():.1f} m, "
      f"90th pct {fx.loc[ok, 'dist_m'].quantile(0.9):.1f} m")
print(f"distinct segments with >=1 usable fix: {fx.loc[ok, 'seg_key'].nunique():,}")

reason
ok                   1361
near_intersection     506
speed_gate            373
heading_mismatch       51 

usable fixes: 1,361 of 2,291 (59%)
snap distance on usable fixes: median 2.0 m, 90th pct 4.2 m
distinct segments with >=1 usable fix: 92


In [4]:
# Gate 0 — Cell 4: from fixes to segments — the Gate 0 scorecard
MIN_FIXES  = 2                                    # usable segment = >= 2 usable fixes (~20+ m of road seen)
band_order = ["Good", "Satisfactory", "Fair", "Poor", "Very Poor", "Serious", "Failed"]

# a new "pass" starts when the segment changes or there is a >5 s break in usable fixes
okf = fx[fx["reason"] == "ok"].copy()
okf["pass_id"] = (okf.index.to_series().diff().gt(5) | okf["seg_key"].ne(okf["seg_key"].shift())).cumsum()
segs = (okf.groupby("seg_key").agg(n_fixes=("reason", "size"), n_passes=("pass_id", "nunique"))
           .join(seg_m.set_index("seg_key")[["rd20full", "pci23", "pci23_d", "cpname", "pav_length"]]))

# repave check: latest repair project end date per segment (one row per segment — no duplication)
pav = pd.read_csv(ROOT / "data/raw/completed_paving_datasd.csv", low_memory=False)
pav["seg_key"] = pav["segment_id"].astype("string").str.strip().str.upper()
last_end = pd.to_datetime(pav["date_end"], errors="coerce").groupby(pav["seg_key"]).max()
segs["last_repair_end"] = segs.index.map(last_end)
segs["repair_2023_or_later"] = segs["last_repair_end"] >= pd.Timestamp("2023-01-01")

usable = segs[(segs["n_fixes"] >= MIN_FIXES) & segs["pci23"].notna()]
bands  = usable["pci23_d"].value_counts().reindex(band_order, fill_value=0)

print(f"segments touched: {len(segs)} | with a PCI: {segs['pci23'].notna().sum()} | "
      f"usable (>= {MIN_FIXES} fixes + PCI): {len(usable)}")
print(f"driven 2+ times (repeat-pass check): {(usable['n_passes'] >= 2).sum()} | "
      f"repair project ending 2023 or later: {usable['repair_2023_or_later'].sum()}")
print(f"PCI of usable segments: mean {usable['pci23'].mean():.1f}, std {usable['pci23'].std():.1f}")
print("usable segments by MIN_FIXES:", {k: int(((segs['n_fixes'] >= k) & segs['pci23'].notna()).sum())
                                         for k in (1, 2, 3, 5)})
print("\n" + bands.to_string())
no_pci = segs[segs["pci23"].isna()]
if len(no_pci):
    print("\ntouched but NO PCI:\n" + no_pci[["rd20full", "n_fixes"]].to_string())

print("\nGATE 0  (thresholds are estimates from the plan)")
for name, passed in {"usable segments >= 300":       len(usable) >= 300,
                     ">= 3 bands with >= 20 each":   (bands >= 20).sum() >= 3,
                     "PCI std >= 10":                usable["pci23"].std() >= 10}.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {name}")

out = ROOT / "data/processed"
segs.to_csv(out / f"gate0_segments_{SESSION}.csv")
fx.drop(columns="geometry", errors="ignore").to_csv(out / f"gate0_fixes_{SESSION}.csv")
print(f"\nsaved: gate0_segments_{SESSION}.csv, gate0_fixes_{SESSION}.csv")

segments touched: 92 | with a PCI: 92 | usable (>= 2 fixes + PCI): 92
driven 2+ times (repeat-pass check): 12 | repair project ending 2023 or later: 22
PCI of usable segments: mean 67.1, std 26.0
usable segments by MIN_FIXES: {1: 92, 2: 92, 3: 92, 5: 92}

pci23_d
Good            30
Satisfactory    25
Fair             8
Poor            10
Very Poor       11
Serious          7
Failed           1

GATE 0  (thresholds are estimates from the plan)
  FAIL  usable segments >= 300
  FAIL  >= 3 bands with >= 20 each
  PASS  PCI std >= 10

saved: gate0_segments_11-11-2026-Pacific-Beach.csv, gate0_fixes_11-11-2026-Pacific-Beach.csv


In [5]:
# Gate 0 — Cell 5: CLEAN yield for this drive + running total over all drives (Viability check V5)
# clean   = has a 2023 PCI AND no repair ending on/after 2023-02-01 (flag built in the cleaning notebook)
# primary = clean AND residential (<= 2 lanes) AND not an alley -> the analysis set
#           (pre-registered 2026-11-12; alley rule "name contains the word ALLEY" frozen 2026-11-13)
cl = pd.read_csv(ROOT / "data/processed/segments_clean.csv",
                 usecols=["iamfloc", "in_clean_set", "is_residential", "is_alley"])
cl["seg_key"] = cl["iamfloc"].astype("string").str.strip().str.upper()
cl = cl.drop_duplicates("seg_key").set_index("seg_key")[["in_clean_set", "is_residential", "is_alley"]]
cl["is_primary"] = ((cl["in_clean_set"] == 1) & (cl["is_residential"] == 1)
                    & (cl["is_alley"] == 0)).astype(int)

def clean_usable(s):
    """usable segments (>= MIN_FIXES usable fixes + a PCI) that are also in the clean pool"""
    u = s[(s["n_fixes"] >= MIN_FIXES) & s["pci23"].notna()].join(cl)
    return u[u["in_clean_set"] == 1]

cu = clean_usable(segs)
print(f"THIS DRIVE ({SESSION}): {len(cu)} clean usable of {len(usable)} usable "
      f"| residential (<= 2 lanes): {int((cu['is_residential'] == 1).sum())} "
      f"| alleys: {int(cu['is_alley'].sum())} | PRIMARY: {int(cu['is_primary'].sum())}")
print(cu.loc[cu["is_primary"] == 1, "pci23_d"].value_counts().reindex(band_order, fill_value=0)
        .rename("primary").to_string())
print("by planning area (primary):", cu.loc[cu["is_primary"] == 1, "cpname"].value_counts().to_dict())

# running total: every drive processed so far (a segment driven on two days counts once)
runs = []
for f in sorted((ROOT / "data/processed").glob("gate0_segments_*.csv")):
    s = pd.read_csv(f, index_col=0)
    s.index = s.index.astype("string").str.strip().str.upper()
    runs.append(clean_usable(s).assign(drive=f.stem.removeprefix("gate0_segments_")))
allc = pd.concat(runs)
allc = allc[~allc.index.duplicated(keep="first")]
prim = allc[allc["is_primary"] == 1]

area_n = allc["cpname"].value_counts()          # clean, per area (pre-registered V5)
area_p = prim["cpname"].value_counts()          # primary, per area
print("\nALL DRIVES SO FAR — per drive:")
print(allc.groupby("drive").agg(clean=("pci23", "size"), alleys=("is_alley", "sum"),
                                primary=("is_primary", "sum")).to_string())
print(f"total clean: {len(allc)} | residential: {int((allc['is_residential'] == 1).sum())} "
      f"| alleys: {int(allc['is_alley'].sum())} | PRIMARY: {len(prim)}")
print("\nPRIMARY set by band and planning area:")
print(pd.crosstab(prim["pci23_d"], prim["cpname"]).reindex(band_order, fill_value=0).to_string())

print("\nVIABILITY V5  (pre-registered: >= 150 clean usable segments across >= 2 planning areas)")
print(f"  {'PASS' if len(allc) >= 150 else 'FAIL'}  clean usable segments >= 150   ({len(allc)})")
print(f"  {'PASS' if (area_n >= 20).sum() >= 2 else 'FAIL'}  >= 2 planning areas (>= 20 each) ({int((area_n >= 20).sum())})")
print("ANALYSIS SET (primary; V6/V7 count only when both lines PASS)")
print(f"  {'PASS' if len(prim) >= 150 else 'FAIL'}  primary segments >= 150        ({len(prim)})")
print(f"  {'PASS' if (area_p >= 20).sum() >= 2 else 'FAIL'}  >= 2 planning areas (>= 20 each) ({int((area_p >= 20).sum())})")
allc.to_csv(ROOT / "data/processed/clean_driven_segments.csv")
print("\nsaved: data/processed/clean_driven_segments.csv  (now includes is_alley, is_primary)")

THIS DRIVE (11-11-2026-Pacific-Beach): 70 clean usable of 92 usable | residential (<= 2 lanes): 66 | alleys: 1 | PRIMARY: 65
pci23_d
Good            28
Satisfactory    13
Fair             5
Poor             3
Very Poor       10
Serious          5
Failed           1
by planning area (primary): {'PACIFIC BEACH': 65}

ALL DRIVES SO FAR — per drive:
                          clean  alleys  primary
drive                                           
10-25-2026-Mira-Mesa         25       0       23
11-11-2026-Pacific-Beach     70       1       65
total clean: 95 | residential: 89 | alleys: 1 | PRIMARY: 88

PRIMARY set by band and planning area:
cpname        MIRA MESA  PACIFIC BEACH
pci23_d                               
Good                  6             28
Satisfactory          4             13
Fair                  4              5
Poor                  4              3
Very Poor             3             10
Serious               1              5
Failed                1              1

VIAB